In [1]:
# | default_exp index_tracking


# Index Tracking
> Inspect, store, and analyze Google Search Console URL indexing status.

In [2]:
# | export
from sqlmodel import Session, select
from seo_rat.models import IndexStatus
from seo_rat.gsc_client import GSCAuth
from googleapiclient.discovery import build
from datetime import datetime
import httpx
from bs4 import BeautifulSoup
import time

from seo_rat.sqlite_db import get_session

In [3]:
# | export
def inspect_url_status(auth: GSCAuth,  # Authenticated GSCAuth instance
                       site_url: str,  # GSC property URL
                       page_url: str  # Page URL to inspect
                       ) -> dict:
    "Inspect URL indexing status from Google Search Console."
    service = build("searchconsole", "v1", credentials=auth.get_credentials())
    response = service.urlInspection().index().inspect(
        body={"inspectionUrl": page_url, "siteUrl": site_url, "languageCode": "en-US"}
    ).execute()
    result = response.get("inspectionResult", {}).get("indexStatusResult", {})
    return {"verdict": result.get("verdict", "UNKNOWN"),
            "coverage_state": result.get("coverageState"),
            "last_crawl_time": result.get("lastCrawlTime"),
            "indexing_state": result.get("indexingState"),
            "robots_txt_state": result.get("robotsTxtState")}


In [4]:
# | test
#| eval: false
from fastcore.test import test_eq

from pprint import pprint
from seo_rat.gsc_client import GSCAuth
from sqlmodel import Session, create_engine, SQLModel
from seo_rat.sqlite_db import get_session

auth = GSCAuth()

# Test inspect_url_status
status = inspect_url_status(auth, "sc-domain:kareemai.com", "https://kareemai.com/")
print(f"Verdict: {status['verdict']}")
test_eq("verdict" in status, True)
pprint(status)

Verdict: PASS
{'coverage_state': 'Submitted and indexed',
 'indexing_state': 'INDEXING_ALLOWED',
 'last_crawl_time': '2026-05-05T21:31:22Z',
 'robots_txt_state': 'ALLOWED',
 'verdict': 'PASS'}


In [5]:
# | export
def store_index_status(session: Session,  # Active database session
                       auth: GSCAuth,  # Authenticated GSCAuth instance
                       site_url: str,  # GSC property URL
                       page_url: str  # Page URL to inspect and store
                       ) -> None:
    "Inspect and store URL index status as a new history row."
    record = IndexStatus(site_url=site_url, page_url=page_url,
                         **inspect_url_status(auth, site_url, page_url))
    session.add(record)
    session.commit()


In [6]:
# | hide
#| eval: false

with get_session() as session:
    store_index_status(session, auth, "sc-domain:kareemai.com", "https://kareemai.com/")
    print("Stored in ./data/seo.db")

    pprint(status)


Stored in ./data/seo.db
{'coverage_state': 'Submitted and indexed',
 'indexing_state': 'INDEXING_ALLOWED',
 'last_crawl_time': '2026-05-05T21:31:22Z',
 'robots_txt_state': 'ALLOWED',
 'verdict': 'PASS'}


In [7]:
# | export
def get_index_status(session: Session,  # Active database session
                     site_url: str,  # GSC property URL
                     verdict: str | None = None  # Optional verdict to filter by
                     ) -> list[IndexStatus]:
    "Get latest index status per page, optionally filtered by verdict."
    from sqlalchemy import func
    latest = (select(IndexStatus.page_url, func.max(IndexStatus.checked_at).label("max_checked"))
              .where(IndexStatus.site_url == site_url)
              .group_by(IndexStatus.page_url).subquery())
    query = select(IndexStatus).join(
        latest, (IndexStatus.page_url == latest.c.page_url) &
                (IndexStatus.checked_at == latest.c.max_checked))
    if verdict: query = query.where(IndexStatus.verdict == verdict)
    return session.exec(query).all()


In [8]:
# | hide
#| eval: false

test_index_status = get_index_status(
    session,
    "sc-domain:kareemai.com",
)
pprint(test_index_status)

[IndexStatus(page_url='https://kareemai.com/papers.html', coverage_state='Submitted and indexed', id=249, indexing_state='INDEXING_ALLOWED', checked_at=datetime.datetime(2026, 5, 5, 19, 58, 10, 543722), verdict='PASS', site_url='sc-domain:kareemai.com', last_crawl_time='2026-04-24T01:04:17Z', robots_txt_state='ALLOWED'),
 IndexStatus(page_url='https://kareemai.com/oss/opensource.html', coverage_state='Submitted and indexed', id=250, indexing_state='INDEXING_ALLOWED', checked_at=datetime.datetime(2026, 5, 5, 19, 58, 18, 242712), verdict='PASS', site_url='sc-domain:kareemai.com', last_crawl_time='2026-02-26T00:21:00Z', robots_txt_state='ALLOWED'),
 IndexStatus(page_url='https://kareemai.com/til/tils/2025-12-15.html', coverage_state='URL is unknown to Google', id=251, indexing_state='INDEXING_STATE_UNSPECIFIED', checked_at=datetime.datetime(2026, 5, 5, 19, 58, 25, 780697), verdict='NEUTRAL', site_url='sc-domain:kareemai.com', last_crawl_time=None, robots_txt_state='ROBOTS_TXT_STATE_UNSPEC

In [10]:
# | export
def get_not_indexed_pages(session: Session,  # Active database session
                          site_url: str  # GSC property URL
                          ) -> list[IndexStatus]:
    "Get pages whose latest index status is not PASS."
    return [p for p in get_index_status(session, site_url) if p.verdict != "PASS"]


In [11]:
#| export
def get_not_indexed_by_reason(session: Session,  # Active database session
                              site_url: str  # GSC property URL
                              ) -> dict[str, list[IndexStatus]]:
    "Group not-indexed pages by their coverage state reason."
    from collections import defaultdict
    grouped = defaultdict(list)
    for page in get_not_indexed_pages(session, site_url):
        grouped[page.coverage_state or "Unknown"].append(page)
    return dict(grouped)

In [12]:
# | export
def fetch_sitemap_urls(sitemap_url: str  # URL of the sitemap XML
                       ) -> list[str]:
    "Fetch all page URLs from a sitemap XML."
    soup = BeautifulSoup(httpx.get(sitemap_url).text, "xml")
    return [loc.text for loc in soup.find_all("loc")]


In [13]:
# | export
def store_all_index_status(session: Session,  # Active database session
                           auth: GSCAuth,  # Authenticated GSCAuth instance
                           site_url: str,  # GSC property URL
                           sitemap_url: str  # URL of the sitemap to process
                           ) -> dict:
    "Check and store index status for all pages in a sitemap."
    urls = fetch_sitemap_urls(sitemap_url)
    results = {"successful": [], "failed": []}
    for i, url in enumerate(urls, 1):
        print(f"Checking {i}/{len(urls)}: {url}")
        try:
            store_index_status(session, auth, site_url, url)
            results["successful"].append(url)
        except Exception as e:
            results["failed"].append({"url": url, "error": str(e)})
        time.sleep(1)
    return results

In [14]:
#| hide
#| eval: false
reasons = get_not_indexed_by_reason(session, "sc-domain:kareemai.com")
for reason, pages in reasons.items():
    print(f"\n{reason} ({len(pages)} pages):")
    for p in pages:
        print(f"  {p.page_url}")
reasons


Duplicate without user-selected canonical (1 pages):
  https://www.alainclean.com/about

URL is unknown to Google (4 pages):
  https://www.alainclean.com/contact
  https://www.alainclean.com/article/عاملات_ابوظبي
  https://www.alainclean.com/article/تنظيف_منازل_ابوظبي
  https://www.alainclean.com/article/تنظيف_منازل_دبي

Crawled - currently not indexed (5 pages):
  https://www.alainclean.com/article/عاملات_دبي
  https://www.alainclean.com/article/عاملات_الشارقة
  https://www.alainclean.com/article/تنظيف_منازل_بني_ياس
  https://www.alainclean.com/article/تنظيف_منازل_الشارقة
  https://alainclean.com/


{'Duplicate without user-selected canonical': [IndexStatus(site_url='sc-domain:alainclean.com', verdict='NEUTRAL', last_crawl_time='2025-12-03T07:50:45Z', robots_txt_state='ALLOWED', page_url='https://www.alainclean.com/about', id=136, coverage_state='Duplicate without user-selected canonical', indexing_state='INDEXING_ALLOWED', checked_at=datetime.datetime(2026, 4, 8, 15, 17, 46, 818752))],
 'URL is unknown to Google': [IndexStatus(site_url='sc-domain:alainclean.com', verdict='NEUTRAL', last_crawl_time=None, robots_txt_state='ROBOTS_TXT_STATE_UNSPECIFIED', page_url='https://www.alainclean.com/contact', id=137, coverage_state='URL is unknown to Google', indexing_state='INDEXING_STATE_UNSPECIFIED', checked_at=datetime.datetime(2026, 4, 8, 15, 17, 54, 458368)),
  IndexStatus(site_url='sc-domain:alainclean.com', verdict='NEUTRAL', last_crawl_time=None, robots_txt_state='ROBOTS_TXT_STATE_UNSPECIFIED', page_url='https://www.alainclean.com/article/عاملات_ابوظبي', id=139, coverage_state='URL 

In [21]:
# | hide
#| eval: false
test_storing_all_index_status = store_all_index_status(
    session,
    auth,
    site_url="sc-domain:kareemai.com",
    sitemap_url="http://127.0.0.1:6287/sitemap.xml",
)
test_storing_all_index_status

Checking 1/58: https://kareemai.com/papers.html
Checking 2/58: https://kareemai.com/oss/opensource.html
Checking 3/58: https://kareemai.com/til/tils/2025-12-15.html
Checking 4/58: https://kareemai.com/til/tils/2025-12-13.html
Checking 5/58: https://kareemai.com/til/tils/2025-06-06-til.html
Checking 6/58: https://kareemai.com/til/tils/2025-05-25-til.html
Checking 7/58: https://kareemai.com/til/tils/2025-05-21-til.html
Checking 8/58: https://kareemai.com/til/tils/2025-05-19-til.html
Checking 9/58: https://kareemai.com/til/tils/2025-05-17-til.html
Checking 10/58: https://kareemai.com/blog/posts/on_device/litert.html
Checking 11/58: https://kareemai.com/blog/posts/speech_recognition/my_dream_job_at_tarteel.html
Checking 12/58: https://kareemai.com/blog/posts/minishlab/ctranslate_maswray.html
Checking 13/58: https://kareemai.com/blog/posts/seo/seo_rat_journey.html
Checking 14/58: https://kareemai.com/blog/posts/seo/how_i_use_nlp_for_seo.html
Checking 15/58: https://kareemai.com/blog/posts/p

{'successful': ['https://kareemai.com/papers.html',
  'https://kareemai.com/oss/opensource.html',
  'https://kareemai.com/til/tils/2025-12-15.html',
  'https://kareemai.com/til/tils/2025-12-13.html',
  'https://kareemai.com/til/tils/2025-06-06-til.html',
  'https://kareemai.com/til/tils/2025-05-25-til.html',
  'https://kareemai.com/til/tils/2025-05-21-til.html',
  'https://kareemai.com/til/tils/2025-05-19-til.html',
  'https://kareemai.com/til/tils/2025-05-17-til.html',
  'https://kareemai.com/blog/posts/on_device/litert.html',
  'https://kareemai.com/blog/posts/speech_recognition/my_dream_job_at_tarteel.html',
  'https://kareemai.com/blog/posts/minishlab/ctranslate_maswray.html',
  'https://kareemai.com/blog/posts/seo/seo_rat_journey.html',
  'https://kareemai.com/blog/posts/seo/how_i_use_nlp_for_seo.html',
  'https://kareemai.com/blog/posts/products_reviews/one_plus_pad_3.html',
  'https://kareemai.com/blog/posts/products_reviews/Huawei_mate_pad_11.html',
  'https://kareemai.com/blog

In [22]:
# | hide
#| eval: false
with get_session() as session:
    test_getting_not_indexed = get_not_indexed_pages(
        session, site_url="sc-domain:kareemai.com"
    )
    pprint(test_getting_not_indexed)

[IndexStatus(site_url='sc-domain:kareemai.com', verdict='NEUTRAL', last_crawl_time=None, robots_txt_state='ROBOTS_TXT_STATE_UNSPECIFIED', page_url='https://kareemai.com/til/tils/2025-12-15.html', id=251, coverage_state='URL is unknown to Google', indexing_state='INDEXING_STATE_UNSPECIFIED', checked_at=datetime.datetime(2026, 5, 5, 19, 58, 25, 780697)),
 IndexStatus(site_url='sc-domain:kareemai.com', verdict='NEUTRAL', last_crawl_time=None, robots_txt_state='ROBOTS_TXT_STATE_UNSPECIFIED', page_url='https://kareemai.com/til/tils/2025-12-13.html', id=252, coverage_state='URL is unknown to Google', indexing_state='INDEXING_STATE_UNSPECIFIED', checked_at=datetime.datetime(2026, 5, 5, 19, 58, 33, 497981)),
 IndexStatus(site_url='sc-domain:kareemai.com', verdict='NEUTRAL', last_crawl_time=None, robots_txt_state='ROBOTS_TXT_STATE_UNSPECIFIED', page_url='https://kareemai.com/til/tils/2025-06-06-til.html', id=253, coverage_state='URL is unknown to Google', indexing_state='INDEXING_STATE_UNSPECIF

In [17]:
#| export
def get_index_history(session: Session,  # Active database session
                      page_url: str  # Page URL to look up
                      ) -> list[IndexStatus]:
    "Get full index status history for a page, newest first."
    return session.exec(
        select(IndexStatus).where(IndexStatus.page_url == page_url)
        .order_by(IndexStatus.checked_at.desc())
    ).all()

In [18]:
#|hide
#| eval: false
history = get_index_history(session, "https://www.smaagarden.com")
for h in history:
    print(f"{h.checked_at} | {h.verdict} | {h.coverage_state}")
